# ЛЦТ: обучение моделей контроля качества денситометрии (Kaggle), версия 4

**Что нового по сравнению с версией 3:** добавлен третий вариант обучения моделей нарушений — **«голова + 5 эпох layer4»**. В версии 3 «только голова» недообучалась (ROC-AUC ниже), а «голова + 15 эпох layer4» переобучалась: `val_loss` в большинстве частей был минимальным не позже 5–10-й эпохи дообучения и ни разу на 15-й. Новый вариант проверяет середину между этими крайностями. Всё остальное — как в версии 3: классификатор зоны, отключённая ось, согласованный `quality_prob`.

**Папку `qc` обновлять не нужно** — изменился только этот ноутбук. Перед запуском: Settings → Accelerator **GPU T4 x2**, Internet **On**.

In [ ]:
!nvidia-smi -L
!pip install -q -U "pydicom>=3.0"

In [ ]:
import glob, os, sys
import pandas as pd

index_hits = glob.glob('/kaggle/input/**/train_index.csv', recursive=True)
assert index_hits, 'train_index.csv не найден в /kaggle/input: добавь датасет через Add Input'
INDEX_CSV = index_hits[0]

qc_hits = glob.glob('/kaggle/input/**/training_utils.py', recursive=True)
assert qc_hits, 'Папка qc не найдена в /kaggle/input'
QC_DIR = os.path.dirname(qc_hits[0])
sys.path.insert(0, QC_DIR)

first_rel = pd.read_csv(INDEX_CSV, keep_default_na=False).rel_path.iloc[0]
DATA_ROOT = None
for root, dirs, files in os.walk('/kaggle/input'):
    if os.path.exists(os.path.join(root, *first_rel.split('/'))):
        DATA_ROOT = root
        break
assert DATA_ROOT, f'Не найден снимок {first_rel}: папка с исследованиями не загрузилась'

WEIGHTS_DIR = '/kaggle/working/weights'
os.makedirs(WEIGHTS_DIR, exist_ok=True)

print('train_index.csv:', INDEX_CSV)
print('qc:            ', QC_DIR)
print('Исследования:  ', DATA_ROOT)
print('Веса будут в:  ', WEIGHTS_DIR)

## Воспроизводимость

Требование ТЗ: одинаковый результат при повторном запуске. Случайность в обучении есть в трёх местах: начальные веса новой головы, порядок снимков в батчах и аугментации. Фиксируем зерно (**seed**) для всех трёх — тогда «случайные» числа каждый раз будут одни и те же.

In [ ]:
import random, time
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score

SEED = 42
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

try:
    from contract import (REGION_LUMBAR_SPINE, REGION_PROXIMAL_FEMUR, DISABLED_VIOLATIONS,
                          REGION_MODEL_CLASSES, REGION_BY_COLUMNS, region_from_size)
    from training_utils import load_region, make_folds, pos_weights, augment, best_thresholds, metrics_table, mean_enabled_auc
    from preprocessing import read_dicom, to_unit_range, to_model_input, to_region_input
    from model import build_model, save_checkpoint, save_region_checkpoint
except ImportError as e:
    raise ImportError(f'В датасете старая версия папки qc ({e}). Загрузи новую версию датасета с обновлёнными файлами qc')
print('Устройство:', DEVICE)
print('Отключённые нарушения:', DISABLED_VIOLATIONS)

## Данные в память

Читать DICOM с диска медленно, а за обучение каждый снимок понадобится сотни раз. Поэтому читаем все снимки **один раз** и держим в памяти уже в виде массивов [0, 1] (Modality LUT → VOI LUT → нормализация — тот же `preprocessing.py`, что и в анализаторе). Паддинг и аугментации делаем на лету.

In [ ]:
def load_images(df):
    images = []
    for i, rel in enumerate(df.rel_path, 1):
        ds = read_dicom(os.path.join(DATA_ROOT, *rel.split('/')))
        images.append(to_unit_range(ds))
        if i % 50 == 0:
            print(f'   прочитано {i}/{len(df)}')
    return images


class QCDataset(Dataset):
    def __init__(self, images, Y, idx, train, seed):
        self.images, self.Y, self.idx, self.train = images, Y, list(idx), train
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, k):
        i = self.idx[k]
        arr = augment(self.images[i], self.rng) if self.train else self.images[i]
        x = to_model_input(np.ascontiguousarray(arr))
        return torch.from_numpy(x), torch.from_numpy(self.Y[i])

## Обучение одной модели: три варианта

| Вариант | Фаза `head` (тело заморожено, lr 1e-3) | Фаза `layer4` (lr 1e-4) | Что показала версия 3 |
|---|---|---|---|
| голова + 15 эпох layer4 | 5 эпох | 15 эпох | лучший ROC-AUC, но `val_loss` растёт — переобучение |
| голова + 5 эпох layer4 | 5 эпох | 5 эпох | новый вариант |
| только голова | 20 эпох | нет | не переобучается, но недообучается: ROC-AUC ниже |

Все три проходят кросс-валидацию на одних и тех же пяти частях. Для финальной модели берётся вариант с лучшим средним ROC-AUC. Вариантов теперь три, поэтому метрики выбранного чуть оптимистичнее, чем при двух; эффект всё ещё небольшой.

In [ ]:
BATCH = 16
VARIANTS = {'голова + 15 эпох layer4': (5, 15), 'голова + 5 эпох layer4': (5, 5), 'только голова': (20, 0)}


def set_trainable(model, phase):
    for name, p in model.named_parameters():
        p.requires_grad = name.startswith('fc.') or (phase == 'layer4' and name.startswith('layer4.'))


def keep_frozen_bn_in_eval(model):
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d) and not any(p.requires_grad for p in m.parameters()):
            m.eval()


def run_loader(model, loader, criterion=None):
    model.eval()
    probs, total = [], 0.0
    with torch.no_grad():
        for x, y in loader:
            logits = model(x.to(DEVICE))
            if criterion is not None:
                total += criterion(logits, y.to(DEVICE)).item() * len(x)
            probs.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(probs), total / len(loader.dataset)


def train_model(images, Y, train_idx, val_idx=None, seed=SEED, freeze_epochs=5, finetune_epochs=15):
    seed_everything(seed)
    model = build_model(Y.shape[1], pretrained=True).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weights(Y[train_idx]), device=DEVICE))
    g = torch.Generator(); g.manual_seed(seed)
    train_loader = DataLoader(QCDataset(images, Y, train_idx, True, seed), batch_size=BATCH, shuffle=True, generator=g)
    val_loader = DataLoader(QCDataset(images, Y, val_idx, False, 0), batch_size=32) if val_idx is not None else None

    for phase, epochs, lr in (('head', freeze_epochs, 1e-3), ('layer4', finetune_epochs, 1e-4)):
        if epochs == 0:
            continue
        set_trainable(model, phase)
        opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-4)
        for epoch in range(1, epochs + 1):
            model.train(); keep_frozen_bn_in_eval(model)
            total = 0.0
            for x, y in train_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                opt.zero_grad()
                loss = criterion(model(x), y)
                loss.backward()
                opt.step()
                total += loss.item() * len(x)
            if val_loader is not None and (epoch % 5 == 0 or epoch == epochs):
                _, val_loss = run_loader(model, val_loader, criterion)
                print(f'      [{phase:6s} {epoch:2d}/{epochs}] train_loss={total / len(train_idx):.3f}  val_loss={val_loss:.3f}')
    return model

## Кросс-валидация, выбор варианта, пороги, финальная модель

Для каждой зоны оба варианта проходят кросс-валидацию **на одних и тех же пяти частях**, чтобы сравнение было честным: разница в результатах — от варианта обучения, а не от того, какие снимки попали в проверку.

**Вариант выбирается по среднему ROC-AUC** включённых нарушений. Не по F1: F1 зависит от порога, а на 6–7 примерах одна картинка сдвигает её на десятки процентов. ROC-AUC смотрит на порядок всех снимков и потому устойчивее.

Честная оговорка: раз мы выбираем лучший из двух вариантов по тем же OOF-предсказаниям, итоговые метрики чуть оптимистичнее, чем будут на новых данных. Вариантов всего два, поэтому эффект небольшой.

In [ ]:
def run_cv(images, Y, folds, freeze_epochs, finetune_epochs):
    P = np.zeros_like(Y)
    n_splits = int(folds.max()) + 1
    for f in range(n_splits):
        tr, va = np.where(folds != f)[0], np.where(folds == f)[0]
        print(f'   Часть {f + 1}/{n_splits}: обучение {len(tr)}, проверка {len(va)}')
        model = train_model(images, Y, tr, va, SEED + f, freeze_epochs, finetune_epochs)
        P[va], _ = run_loader(model, DataLoader(QCDataset(images, Y, va, False, 0), batch_size=32))
    return P


def run_region(region, short_name, n_splits=5):
    df, Y, names = load_region(INDEX_CSV, region)
    print(f'=== {region}: {len(df)} снимков, нарушений: {dict(zip(names, Y.sum(0).astype(int).tolist()))}')
    images = load_images(df)
    folds = make_folds(df, Y, n_splits, SEED)

    results = {}
    for variant, (fe, ft) in VARIANTS.items():
        print(f'\n--- Вариант «{variant}» ---')
        results[variant] = run_cv(images, Y, folds, fe, ft)

    comparison = pd.DataFrame(
        {v: [roc_auc_score(Y[:, j], P[:, j]) for j in range(len(names))] + [mean_enabled_auc(Y, P, names)]
         for v, P in results.items()},
        index=[n + (' (отключено)' if n in DISABLED_VIOLATIONS else '') for n in names] + ['СРЕДНЕЕ по включённым'],
    ).round(3)
    print('\nROC-AUC по out-of-fold предсказаниям:')
    display(comparison)
    best = max(results, key=lambda v: mean_enabled_auc(Y, results[v], names))
    print(f'Выбран вариант: «{best}»')

    P = results[best]
    thresholds = best_thresholds(Y, P, names)
    table = metrics_table(Y, P, names, thresholds)
    print('\nМетрики выбранного варианта:')
    display(table)

    print('Финальная модель на всех снимках...')
    fe, ft = VARIANTS[best]
    final = train_model(images, Y, np.arange(len(df)), None, SEED, fe, ft)
    path = f'{WEIGHTS_DIR}/{short_name}.pt'
    save_checkpoint(path, final, region, names, thresholds,
                    metrics={'variant': best, 'table': table.to_dict(orient='records'), 'auc_by_variant': comparison.to_dict()})

    oof = pd.DataFrame(P, columns=names)
    for j, n in enumerate(names):
        oof[f'эксперт: {n}'] = Y[:, j].astype(int)
    oof = oof.assign(study_uid=df.study_uid.values, rel_path=df.rel_path.values,
                     rows=[im.shape[0] for im in images], fold=folds)
    oof.to_csv(f'{WEIGHTS_DIR}/oof_{short_name}.csv', index=False)
    print(f'Сохранено: {path}')
    return comparison, table

In [ ]:
t0 = time.time()
spine_comparison, spine_table = run_region(REGION_LUMBAR_SPINE, 'spine')
print(f'\nПозвоночник готов за {(time.time() - t0) / 60:.1f} мин')

In [ ]:
t0 = time.time()
femur_comparison, femur_table = run_region(REGION_PROXIMAL_FEMUR, 'femur')
print(f'\nБедро готово за {(time.time() - t0) / 60:.1f} мин')

## Классификатор зоны

Третья модель отвечает на один вопрос: позвоночник на снимке или бедро. Она нужна не вместо правила по ширине, а **рядом с ним**:
- если правило и классификатор согласны — всё как раньше;
- если расходятся — зона остаётся по правилу, но анализатор записывает предупреждение;
- если ширина нестандартная (правило не знает ответа) — зону определяет классификатор, если уверен не меньше чем на 90%.

**Вход — квадрат 224×224, а не родная ширина.** Это главное решение здесь. Если бы классификатор видел исходную ширину снимка (300 или 280), ему было бы проще всего выучить само правило «300 — позвоночник», а не анатомию. Тогда в том единственном случае, ради которого он нужен — снимок позвоночника шириной 280 — он ошибся бы вместе с правилом, и проверка не сработала бы. Растянув все снимки в один квадрат, мы убираем ширину из того, что сеть может увидеть. Искажение пропорций здесь не мешает: отличить позвонки от бедренной кости можно и на слегка растянутой картинке.

Задача лёгкая, поэтому учим только голову (10 эпох). Проверяем той же кросс-валидацией на 5 частей с группировкой по исследованию. Размеченных снимков для зоны больше, чем для нарушений: сюда идут все 249, включая три спорные строки — нарушения там спорные, а зона нет.

In [ ]:
REGION_EPOCHS = 10


class RegionDataset(Dataset):
    def __init__(self, images, y, idx, train, seed):
        self.images, self.y, self.idx, self.train = images, y, list(idx), train
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, k):
        i = self.idx[k]
        arr = augment(self.images[i], self.rng) if self.train else self.images[i]
        return torch.from_numpy(to_region_input(np.ascontiguousarray(arr))), int(self.y[i])


def train_region_model(images, y, train_idx, seed=SEED):
    seed_everything(seed)
    model = build_model(len(REGION_MODEL_CLASSES), pretrained=True).to(DEVICE)
    set_trainable(model, 'head')
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    g = torch.Generator(); g.manual_seed(seed)
    loader = DataLoader(RegionDataset(images, y, train_idx, True, seed), batch_size=BATCH, shuffle=True, generator=g)
    for epoch in range(REGION_EPOCHS):
        model.train(); keep_frozen_bn_in_eval(model)
        for x, t in loader:
            x, t = x.to(DEVICE), t.to(DEVICE)
            opt.zero_grad()
            loss = criterion(model(x), t)
            loss.backward()
            opt.step()
    return model


def predict_region(model, images, y, idx):
    model.eval()
    loader = DataLoader(RegionDataset(images, y, idx, False, 0), batch_size=32)
    probs = []
    with torch.no_grad():
        for x, _ in loader:
            probs.append(torch.softmax(model(x.to(DEVICE)), dim=1).cpu().numpy())
    return np.concatenate(probs)


def run_region_classifier(n_splits=5):
    df = pd.read_csv(INDEX_CSV, keep_default_na=False)
    df = df[df.region.isin(REGION_MODEL_CLASSES)].reset_index(drop=True)
    y = np.array([REGION_MODEL_CLASSES.index(r) for r in df.region])
    print(f'=== Классификатор зоны: {len(df)} снимков', dict(zip(REGION_MODEL_CLASSES, np.bincount(y).tolist())))
    images = load_images(df)
    folds = make_folds(df, np.eye(len(REGION_MODEL_CLASSES), dtype=np.float32)[y], n_splits, SEED)

    P = np.zeros((len(df), len(REGION_MODEL_CLASSES)), dtype=np.float32)
    for f in range(n_splits):
        tr, va = np.where(folds != f)[0], np.where(folds == f)[0]
        print(f'   Часть {f + 1}/{n_splits}: обучение {len(tr)}, проверка {len(va)}')
        model = train_region_model(images, y, tr, SEED + f)
        P[va] = predict_region(model, images, y, va)

    pred = P.argmax(1)
    names = np.array(REGION_MODEL_CLASSES)
    print(f'\nТочность по out-of-fold предсказаниям: {(pred == y).mean():.3f} ({(pred == y).sum()} из {len(y)})')
    display(pd.crosstab(pd.Series(names[y], name='эксперт'), pd.Series(names[pred], name='классификатор')))
    conf = P.max(1)
    print(f'Минимальная уверенность классификатора: {conf.min():.3f}; снимков с уверенностью ниже 0,9: {(conf < 0.9).sum()}')
    wrong = df[pred != y]
    if len(wrong):
        print('Ошибки классификатора:')
        display(wrong[['study_uid', 'rel_path', 'region']].assign(уверенность=conf[pred != y]))
    rule = np.array([region_from_size(im.shape[1]) for im in images])
    print(f'Согласие с правилом по ширине: {(rule == names[pred]).mean():.3f}')

    print('\nФинальный классификатор зоны на всех снимках...')
    final = train_region_model(images, y, np.arange(len(df)), SEED)
    path = f'{WEIGHTS_DIR}/region.pt'
    save_region_checkpoint(path, final, REGION_MODEL_CLASSES,
                           metrics={'oof_accuracy': float((pred == y).mean()), 'min_confidence': float(conf.min())})
    print(f'Сохранено: {path}')

In [ ]:
t0 = time.time()
run_region_classifier()
print(f'\nКлассификатор зоны готов за {(time.time() - t0) / 60:.1f} мин')

## Проверка анализатора для бэкенда

Загружаем только что сохранённые веса **тем же кодом, которым будет пользоваться бэкенд**, на процессоре (как в работе), и дважды анализируем одно исследование. Результаты должны совпасть до последней цифры — это и есть проверка воспроизводимости.

In [ ]:
from analyzer import QualityAnalyzer

analyzer = QualityAnalyzer(WEIGHTS_DIR, device='cpu')
print('Классификатор зоны загружен:', analyzer.region_model is not None)
study = os.path.join(DATA_ROOT, sorted(os.listdir(DATA_ROOT))[0])
rows1 = analyzer.analyze_study(study)
rows2 = analyzer.analyze_study(study)

cols = ['anatomical_region', 'quality_class', 'quality_prob', 'violation_type', 'processing_status', 'time_of_processing']
display(pd.DataFrame(rows1)[cols])
same = [(r['quality_prob'], r['violation_type']) for r in rows1] == [(r['quality_prob'], r['violation_type']) for r in rows2]
print('Повторный запуск дал тот же результат:', same)
print('Предупреждений:', len(analyzer.warnings))

## Нестандартные снимки

В обучающих данных есть два снимка шириной 248 (исследование с эндопротезом). Правило по ширине для них ответа не знает. Посмотрим, что с ними сделает анализатор теперь: какую зону предложит классификатор и насколько он уверен. Эти снимки классификатор при обучении не видел — их нет в разметке.

In [ ]:
import pydicom
odd_studies = set()
for root, dirs, files in os.walk(DATA_ROOT):
    for f in files:
        p = os.path.join(root, f)
        try:
            ds = pydicom.dcmread(p, stop_before_pixels=True)
        except Exception:
            continue
        if int(ds.Columns) not in REGION_BY_COLUMNS:
            odd_studies.add(os.path.relpath(p, DATA_ROOT).split(os.sep)[0])

print('Исследований с нестандартными снимками:', len(odd_studies))
for s in sorted(odd_studies):
    rows = analyzer.analyze_study(os.path.join(DATA_ROOT, s))
    print(f'\n{s}')
    display(pd.DataFrame(rows)[['anatomical_region', 'quality_class', 'violation_type', 'processing_status']])
    if analyzer.warnings:
        display(pd.DataFrame(analyzer.warnings)[['rows', 'columns', 'zone_by_classifier', 'classifier_confidence', 'event']])

## Готово

В `/kaggle/working/weights/` лежат:
- `spine.pt`, `femur.pt` — модели нарушений;
- `region.pt` — классификатор зоны (анализатор подхватывает его сам, если файл лежит рядом с остальными);
- `oof_spine.csv`, `oof_femur.csv` — out-of-fold предсказания с ответами эксперта.

Следующая ячейка упаковывает их в `weights.zip` (панель **Output** справа). Когда закончишь, останови сессию, чтобы не расходовать квоту GPU.

In [ ]:
!cd /kaggle/working && zip -r -q weights.zip weights && ls -la /kaggle/working